# 📊 TSRL — Data Exploration

Fetch OHLCV data, visualize price action, and analyze return distributions.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime

from src.application.services.data_service import DataService

plt.style.use('dark_background')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['figure.dpi'] = 100

## 1. Fetch Data

In [ ]:
data_service = DataService()

symbols = ['AAPL', 'GOOGL', 'MSFT']
start = datetime(2022, 1, 1)
end = datetime(2024, 12, 31)

data = {}
for symbol in symbols:
    df, source = data_service.fetch_data(symbol, start, end)
    data[symbol] = df
    print(f'{symbol}: {len(df)} bars ({source})')

data['AAPL'].tail()

## 2. Price Chart

In [ ]:
fig, ax = plt.subplots()

for symbol in symbols:
    close = data[symbol]['close']
    normalized = close / close.iloc[0] * 100
    ax.plot(normalized.index, normalized.values, label=symbol, linewidth=1.5)

ax.set_title('Normalized Price (Base 100)', fontsize=14, fontweight='bold')
ax.set_ylabel('Price (normalized)')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 3. Return Distribution

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, symbol in zip(axes, symbols):
    returns = data[symbol]['close'].pct_change().dropna()
    ax.hist(returns, bins=50, alpha=0.7, color='#4fc3f7', edgecolor='white', linewidth=0.5)
    ax.axvline(returns.mean(), color='#ff5252', linestyle='--', label=f'Mean: {returns.mean():.4f}')
    ax.set_title(f'{symbol} Daily Returns', fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Summary Statistics

In [ ]:
from src.analytics.risk_metrics import RiskMetricsCalculator

stats = []
for symbol in symbols:
    returns = data[symbol]['close'].pct_change().dropna()
    stats.append({
        'Symbol': symbol,
        'Total Return': f"{(data[symbol]['close'].iloc[-1] / data[symbol]['close'].iloc[0] - 1) * 100:.2f}%",
        'Annualized Vol': f"{returns.std() * np.sqrt(252) * 100:.2f}%",
        'Sharpe Ratio': f"{RiskMetricsCalculator.calculate_sharpe_ratio(returns):.2f}",
        'Max Drawdown': f"{RiskMetricsCalculator.calculate_max_drawdown(returns):.2f}%",
        'Skewness': f"{returns.skew():.2f}",
        'Kurtosis': f"{returns.kurtosis():.2f}",
    })

pd.DataFrame(stats).set_index('Symbol')

## 5. Volatility Analysis

In [ ]:
fig, ax = plt.subplots()

for symbol in symbols:
    returns = data[symbol]['close'].pct_change().dropna()
    rolling_vol = returns.rolling(21).std() * np.sqrt(252) * 100
    ax.plot(rolling_vol.index, rolling_vol.values, label=symbol, linewidth=1.2)

ax.set_title('21-Day Rolling Volatility (Annualized)', fontsize=14, fontweight='bold')
ax.set_ylabel('Volatility (%)')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()